## Step 1: Install Required Packages

First, we need to install CrewAI and its dependencies. We'll use:
- `crewai`: The main framework (includes the `@tool` decorator we use for custom tools)
- `python-dotenv`: Loads credentials from a `.env` file

**Note:** CrewAI currently requires Python 3.10–3.13 and pins the MCP SDK to 1.x, which conflicts with the LiteLLM version used in the Google ADK notebook. Run this notebook in its own environment/kernel.

In [12]:
# Install required packages
%pip install -q "crewai>=1.15" python-dotenv

# Windows only: pywin32 (used by the MCP SDK) normally needs a kernel restart after installation.
# Re-reading the site-packages .pth files makes it importable right away.
import site, sys, sysconfig
if sys.platform == "win32":
    for _site_dir in (sysconfig.get_paths()["purelib"], site.getusersitepackages()):
        site.addsitedir(_site_dir)

Note: you may need to restart the kernel to use updated packages.


## Step 2: Setup API Keys and Environment

To use Azure OpenAI, you need your resource's API key, endpoint, and model deployment name (Azure portal or Azure AI Foundry → your resource → **Keys and Endpoint**).

**Security Best Practice:** Store your API key in a `.env` file, not in the code! The cell below loads it from `.env` (in this folder or any parent folder) and prompts for anything missing.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [13]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## Step 3: Create Custom Tools

Tools are functions that agents can use to perform specific tasks. Let's create custom tools for our movie production assistant.

### Tool 1: Script Sentiment Analyzer
Analyzes the emotional tone of a script excerpt.

In [14]:
from crewai.tools import tool
import re

@tool("Script Sentiment Analyzer")
def analyze_script_sentiment(script_text: str) -> str:
    """
    Analyzes the sentiment and emotional tone of a movie script excerpt.
    Returns sentiment score and dominant emotions.

    Args:
        script_text: A string containing the script excerpt to analyze
    """
    # Simple sentiment analysis based on keywords
    positive_words = ['love', 'happy', 'joy', 'wonderful', 'amazing', 'brilliant', 'triumph', 'success']
    negative_words = ['hate', 'sad', 'angry', 'terrible', 'awful', 'tragedy', 'failure', 'death']
    suspense_words = ['mystery', 'unknown', 'hidden', 'secret', 'suspense', 'thriller']

    script_lower = script_text.lower()

    pos_count = sum(1 for word in positive_words if word in script_lower)
    neg_count = sum(1 for word in negative_words if word in script_lower)
    sus_count = sum(1 for word in suspense_words if word in script_lower)

    total = pos_count + neg_count + sus_count

    if total == 0:
        return "Sentiment: Neutral - No strong emotional indicators detected."

    result = f"Sentiment Analysis Results:\n"
    result += f"- Positive tone: {pos_count}/{total} ({pos_count/total*100:.1f}%)\n"
    result += f"- Negative tone: {neg_count}/{total} ({neg_count/total*100:.1f}%)\n"
    result += f"- Suspenseful tone: {sus_count}/{total} ({sus_count/total*100:.1f}%)\n"

    if pos_count > neg_count and pos_count > sus_count:
        result += "\nOverall: Uplifting and positive story"
    elif neg_count > pos_count and neg_count > sus_count:
        result += "\nOverall: Dark and dramatic narrative"
    else:
        result += "\nOverall: Tense and mysterious atmosphere"

    return result

# Test the tool
test_script = "A tale of love and triumph as Sarah discovers the hidden mystery that brings her joy."
print(analyze_script_sentiment.run(test_script))

Sentiment Analysis Results:
- Positive tone: 3/5 (60.0%)
- Negative tone: 0/5 (0.0%)
- Suspenseful tone: 2/5 (40.0%)

Overall: Uplifting and positive story


### Tool 2: Budget Calculator
Calculates production costs based on different categories.

In [15]:
@tool("Production Budget Calculator")
def calculate_production_budget(category: str, days: int = 30, crew_size: int = 50) -> str:
    """
    Calculates estimated production budget for different movie categories.

    Args:
        category: Type of production (indie, medium, blockbuster)
        days: Number of shooting days
        crew_size: Number of crew members
    """
    # Base daily rates per category
    rates = {
        'indie': {'daily': 50000, 'crew': 500, 'equipment': 10000},
        'medium': {'daily': 200000, 'crew': 2000, 'equipment': 50000},
        'blockbuster': {'daily': 1000000, 'crew': 10000, 'equipment': 250000}
    }

    category = category.lower()
    if category not in rates:
        return f"Unknown category. Choose from: {', '.join(rates.keys())}"

    rate = rates[category]

    production_cost = rate['daily'] * days
    crew_cost = rate['crew'] * crew_size * days
    equipment_cost = rate['equipment'] * days
    contingency = (production_cost + crew_cost + equipment_cost) * 0.15

    total = production_cost + crew_cost + equipment_cost + contingency

    result = f"📊 Budget Breakdown for {category.upper()} Production:\n"
    result += f"\n🎬 Production Costs: ${production_cost:,}"
    result += f"\n👥 Crew Costs ({crew_size} members): ${crew_cost:,}"
    result += f"\n📹 Equipment Rental: ${equipment_cost:,}"
    result += f"\n🛡️ Contingency (15%): ${contingency:,}"
    result += f"\n\n💰 TOTAL ESTIMATED BUDGET: ${total:,}"
    result += f"\n📅 For {days} shooting days"

    return result

# Test the tool
print(calculate_production_budget.run("medium", days=45, crew_size=75))

📊 Budget Breakdown for MEDIUM Production:

🎬 Production Costs: $9,000,000
👥 Crew Costs (75 members): $6,750,000
📹 Equipment Rental: $2,250,000
🛡️ Contingency (15%): $2,700,000.0

💰 TOTAL ESTIMATED BUDGET: $20,700,000.0
📅 For 45 shooting days


### Tool 3: Soundtrack Genre Recommender
Suggests music genres based on the movie's theme and mood.

In [16]:
@tool("Soundtrack Genre Recommender")
def recommend_soundtrack_genre(movie_genre: str, mood: str) -> str:
    """
    Recommends soundtrack genres based on movie genre and mood.

    Args:
        movie_genre: The genre of the movie (action, drama, comedy, horror, sci-fi, romance)
        mood: The desired mood (intense, light, emotional, mysterious, upbeat)
    """
    recommendations = {
        'action': {
            'intense': ['Epic Orchestral', 'Electronic/Synth', 'Heavy Metal'],
            'light': ['Pop Rock', 'Funk', 'Electronic Pop'],
            'emotional': ['Cinematic Strings', 'Piano & Strings', 'Alternative Rock'],
        },
        'drama': {
            'intense': ['Classical', 'Dramatic Orchestral', 'Jazz Noir'],
            'emotional': ['Piano Solo', 'String Quartet', 'Acoustic Guitar'],
            'mysterious': ['Ambient', 'Minimalist Piano', 'Contemporary Classical'],
        },
        'comedy': {
            'upbeat': ['Jazz', 'Pop', 'Ska', 'Funk'],
            'light': ['Acoustic Pop', 'Ukulele', 'Whistling & Bells'],
        },
        'horror': {
            'intense': ['Dark Ambient', 'Industrial', 'Atonal Orchestral'],
            'mysterious': ['Theremin', 'Dissonant Strings', 'Electronic Drone'],
        },
        'sci-fi': {
            'intense': ['Electronic/Synth', 'Cyberpunk', 'Orchestral Hybrid'],
            'mysterious': ['Ambient Electronic', 'Experimental', 'Synth Pad'],
        },
        'romance': {
            'emotional': ['Classical Piano', 'Acoustic', 'Indie Folk'],
            'light': ['Soft Pop', 'Bossa Nova', 'Acoustic Guitar'],
            'upbeat': ['Pop', 'Soul', 'Light Jazz'],
        }
    }

    genre = movie_genre.lower()
    mood_key = mood.lower()

    if genre not in recommendations:
        return f"Unknown genre. Try: {', '.join(recommendations.keys())}"

    genre_moods = recommendations[genre]
    if mood_key not in genre_moods:
        return f"For {genre}, available moods are: {', '.join(genre_moods.keys())}"

    suggested = genre_moods[mood_key]

    result = f"🎵 Soundtrack Recommendations for {genre.upper()} ({mood} mood):\n\n"
    for i, genre_rec in enumerate(suggested, 1):
        result += f"{i}. {genre_rec}\n"

    result += f"\n💡 Tip: Consider mixing {suggested[0]} with {suggested[-1]} for dynamic scenes!"

    return result

# Test the tool
print(recommend_soundtrack_genre.run("sci-fi", "mysterious"))

🎵 Soundtrack Recommendations for SCI-FI (mysterious mood):

1. Ambient Electronic
2. Experimental
3. Synth Pad

💡 Tip: Consider mixing Ambient Electronic with Synth Pad for dynamic scenes!


## Step 4: Integrate MCP Tool (Model Context Protocol)

MCP tools allow agents to interact with external systems and APIs. Here we'll create a simulated MCP tool for accessing a movie database.

**Note:** In production, MCP tools connect to real external services. This is a simplified demonstration.

In [17]:
from typing import Dict, Any

@tool("Movie Market Research MCP")
def fetch_market_data(genre: str) -> str:
    """
    Fetches market research data for a specific movie genre from the database.
    This simulates an MCP tool that connects to external market research APIs.

    Args:
        genre: The movie genre to research (action, drama, comedy, horror, sci-fi, romance)
    """
    # Simulated market data (in production, this would call a real API)
    market_database = {
        'action': {
            'avg_box_office': '$450M',
            'audience_demographic': '18-35 years, 65% male',
            'peak_season': 'Summer (May-August)',
            'trending_themes': ['Superheroes', 'International espionage', 'Disaster scenarios'],
            'streaming_performance': 'High - 85% completion rate',
        },
        'drama': {
            'avg_box_office': '$120M',
            'audience_demographic': '30-60 years, 55% female',
            'peak_season': 'Awards season (October-February)',
            'trending_themes': ['Social justice', 'Biography', 'Family dynamics'],
            'streaming_performance': 'Medium - 72% completion rate',
        },
        'comedy': {
            'avg_box_office': '$180M',
            'audience_demographic': '18-45 years, balanced gender',
            'peak_season': 'Year-round, slight spike in holidays',
            'trending_themes': ['Romantic comedy', 'Workplace humor', 'Cultural clash'],
            'streaming_performance': 'Very High - 90% completion rate',
        },
        'horror': {
            'avg_box_office': '$95M',
            'audience_demographic': '16-30 years, 52% male',
            'peak_season': 'Fall (September-November)',
            'trending_themes': ['Psychological horror', 'Folk horror', 'Home invasion'],
            'streaming_performance': 'High - 82% completion rate',
        },
        'sci-fi': {
            'avg_box_office': '$380M',
            'audience_demographic': '18-40 years, 60% male',
            'peak_season': 'Summer and Holiday season',
            'trending_themes': ['AI & technology', 'Space exploration', 'Time travel'],
            'streaming_performance': 'High - 88% completion rate',
        },
        'romance': {
            'avg_box_office': '$85M',
            'audience_demographic': '18-50 years, 70% female',
            'peak_season': 'Valentine\'s Day season (January-February)',
            'trending_themes': ['Second chance romance', 'Holiday romance', 'Cross-cultural love'],
            'streaming_performance': 'Very High - 92% completion rate',
        },
    }

    genre_key = genre.lower()

    if genre_key not in market_database:
        return f"❌ No market data available for '{genre}'. Available genres: {', '.join(market_database.keys())}"

    data = market_database[genre_key]

    result = f"📈 Market Research Report: {genre.upper()} Genre\n"
    result += f"\n💰 Average Box Office: {data['avg_box_office']}"
    result += f"\n👥 Target Demographic: {data['audience_demographic']}"
    result += f"\n📅 Best Release Window: {data['peak_season']}"
    result += f"\n🎬 Trending Themes:"
    for theme in data['trending_themes']:
        result += f"\n   • {theme}"
    result += f"\n📺 Streaming Performance: {data['streaming_performance']}"
    result += f"\n\n✅ Data retrieved from Market Intelligence Database (MCP)"

    return result

# Test the MCP tool
print(fetch_market_data.run("action"))

📈 Market Research Report: ACTION Genre

💰 Average Box Office: $450M
👥 Target Demographic: 18-35 years, 65% male
📅 Best Release Window: Summer (May-August)
🎬 Trending Themes:
   • Superheroes
   • International espionage
   • Disaster scenarios
📺 Streaming Performance: High - 85% completion rate

✅ Data retrieved from Market Intelligence Database (MCP)


## Step 5: Configure the Language Model

Now let's set up our Azure OpenAI deployment to power our agents. Azure OpenAI's v1 API is OpenAI-compatible, so we use CrewAI's native OpenAI provider (`openai/<deployment>`) with `base_url` pointing at the Azure endpoint.

**Note:** GPT-5-family deployments are reasoning models that only support the default `temperature`, so we don't set it.

In [18]:
from crewai import LLM

# Initialize the Azure OpenAI deployment using CrewAI's LLM wrapper
llm = LLM(
    model=f"openai/{AZURE_OPENAI_DEPLOYMENT}",
    base_url=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print("✅ Language model configured: Azure OpenAI")
print(f"   - Deployment: {AZURE_OPENAI_DEPLOYMENT}")

✅ Language model configured: Azure OpenAI
   - Deployment: gpt-5.4-mini


## Step 6: Create Specialized Agents

Agents are AI assistants with specific roles and tools. Let's create a crew of specialized agents for movie production:

1. **Script Analyst** - Analyzes scripts and provides creative feedback
2. **Budget Manager** - Handles financial planning and budgeting
3. **Market Researcher** - Provides market insights and recommendations

In [19]:
from crewai import Agent

# Agent 1: Script Analyst
script_analyst = Agent(
    role="Script Analyst",
    goal="Analyze movie scripts for sentiment, themes, and recommend appropriate soundtracks",
    backstory="""You are an experienced script analyst with 15 years in Hollywood.
    You have a keen eye for emotional depth and understand how music enhances storytelling.
    You've worked on numerous award-winning films and know what makes a script compelling.""",
    tools=[analyze_script_sentiment, recommend_soundtrack_genre],
    llm=llm,
    verbose=True,  # Shows detailed thinking process
    allow_delegation=False,  # Can't delegate tasks to other agents
)

# Agent 2: Budget Manager
budget_manager = Agent(
    role="Production Budget Manager",
    goal="Calculate accurate production budgets and optimize spending across departments",
    backstory="""You are a meticulous financial expert specializing in film production.
    With an MBA and 10 years of experience managing budgets for indie to blockbuster films,
    you ensure projects stay financially viable while maintaining creative vision.""",
    tools=[calculate_production_budget],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Agent 3: Market Researcher
market_researcher = Agent(
    role="Film Market Research Analyst",
    goal="Provide data-driven insights about market trends, audience preferences, and optimal release strategies",
    backstory="""You are a data scientist specializing in entertainment industry analytics.
    You analyze box office trends, streaming data, and audience demographics to help studios
    make informed decisions about production and distribution.""",
    tools=[fetch_market_data],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print("✅ Created 3 specialized agents:")
print("   1. Script Analyst (sentiment & soundtrack)")
print("   2. Budget Manager (financial planning)")
print("   3. Market Researcher (market intelligence)")

✅ Created 3 specialized agents:
   1. Script Analyst (sentiment & soundtrack)
   2. Budget Manager (financial planning)
   3. Market Researcher (market intelligence)


## Step 7: Define Tasks for Agents

Tasks are specific assignments given to agents. Each task has:
- A clear description
- An assigned agent
- Expected output format

In [20]:
from crewai import Task

# Sample movie concept for our tasks
movie_concept = """
Title: "Echoes of Tomorrow"
Genre: Sci-Fi Thriller
Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals
humanity's tragic future. Racing against time and a mysterious organization, she must decide
whether to alter the timeline or accept humanity's fate.

Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both
triumph and terror. The hidden secret she uncovers could save millions, but the mystery
of who left this message haunts her every waking moment."
"""

# Task 1: Analyze the script
task_analyze_script = Task(
    description=f"""Analyze the following movie concept and script excerpt:

    {movie_concept}

    Use your tools to:
    1. Analyze the sentiment of the script excerpt
    2. Recommend appropriate soundtrack genres based on the genre and mood
    3. Provide creative suggestions for enhancing the emotional impact
    """,
    expected_output="A detailed analysis including sentiment breakdown, soundtrack recommendations, and creative suggestions",
    agent=script_analyst,
)

# Task 2: Calculate budget
task_budget_calculation = Task(
    description="""Based on the sci-fi thriller movie concept, calculate the production budget.

    Assume:
    - This is a MEDIUM budget production
    - 60 days of shooting
    - Crew size of 85 people

    Provide a detailed breakdown and recommendations for budget optimization.
    """,
    expected_output="Complete budget breakdown with optimization recommendations",
    agent=budget_manager,
)

# Task 3: Market research
task_market_research = Task(
    description="""Conduct market research for the sci-fi thriller genre.

    Use the MCP tool to fetch market data and provide:
    1. Current market performance of sci-fi films
    2. Target audience insights
    3. Optimal release timing
    4. Strategic recommendations based on trending themes
    """,
    expected_output="Comprehensive market analysis with strategic release recommendations",
    agent=market_researcher,
)

print("✅ Created 3 tasks:")
print("   1. Script Analysis Task")
print("   2. Budget Calculation Task")
print("   3. Market Research Task")

✅ Created 3 tasks:
   1. Script Analysis Task
   2. Budget Calculation Task
   3. Market Research Task


## Step 8: Assemble the Crew and Execute

Now we bring it all together! A Crew coordinates multiple agents working on related tasks.

**Process Types:**
- `sequential`: Tasks execute one after another (used here)
- `hierarchical`: A manager agent coordinates other agents

**Note:** Jupyter already runs an asyncio event loop, and CrewAI refuses to run synchronously inside one. In notebooks we therefore use `await crew.kickoff_async()`; in a regular Python script, use `crew.kickoff()`.

In [21]:
from crewai import Crew, Process

# Assemble the crew
movie_production_crew = Crew(
    agents=[script_analyst, budget_manager, market_researcher],
    tasks=[task_analyze_script, task_budget_calculation, task_market_research],
    process=Process.sequential,  # Tasks run in order
    verbose=True,  # Enable detailed output for learning
)

print("✅ Crew assembled with 3 agents and 3 tasks")
print("\n🚀 Starting production analysis...\n")
print("=" * 80)

# Execute the crew's tasks
result = await movie_production_crew.kickoff_async()

print("\n" + "=" * 80)
print("\n✅ Analysis Complete!\n")

✅ Crew assembled with 3 agents and 3 tasks

🚀 Starting production analysis...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 052c0360-d22b-4dec-9bb0-f3912b907b0f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│  ID: 738cfde3-41da-4f4f-849d-86be06f973f4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze the following movie concept and script excerpt:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: "Echoes of Tomorrow"                                                                                    │
│  Genre: Sci-Fi Thriller                                                                                         │
│  Logline: A brilliant AI researcher discovers a hidden message in quantum data that reveals                     │
│  humanity's tragic future. Racing against time and a mysterious organization, she must decide                   │
│  whether to alter the timeline or accept humanity's fate.                                                       │
│                                                                                                                 │
│  Script Excerpt: "In the depths of the unknown laboratory, Sarah's discovery brings both                        │
│  triumph and terror. The hidden secret she uncovers could save millions, but the mystery                        │
│  of who left this message haunts her every waking moment."                                                      │
│                                                                                                                 │
│                                                                                                                 │
│      Use your tools to:                                                                                         │
│      1. Analyze the sentiment of the script excerpt                                                             │
│      2. Recommend appropriate soundtrack genres based on the genre and mood                                     │
│      3. Provide creative suggestions for enhancing the emotional impact                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: script_sentiment_analyzer                                                                                │
│  Args: {'script_text': "In the depths of the unknown laboratory, Sarah's discovery brings both triumph and      │
│  terror. The hidden secret she uncovers could save millions, but the mystery of who left this message...        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: script_sentiment_analyzer                                                                                │
│  Output: Sentiment Analysis Results:                                                                            │
│  - Positive tone: 1/5 (20.0%)                                                                                   │
│  - Negative tone: 0/5 (0.0%)                                                                                    │
│  - Suspenseful tone: 4/5 (80.0%)                                                                                │
│                                                                                                                 │
│  Overall: Tense and mysterious atmosphere                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (18 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Conduct market research for the sci-fi thriller genre.                                                   │
│                                                                                                                 │
│      Use the MCP tool to fetch market data and provide:                                                         │
│      1. Current market performance of sci-fi films                                                              │
│      2. Target audience insights                                                                                │
│      3. Optimal release timing                                                                                  │
│      4. Strategic recommendations based on trending themes                                                      │
│                                                                                                                 │
│  Agent: Film Market Research Analyst                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



✅ Analysis Complete!



╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 052c0360-d22b-4dec-9bb0-f3912b907b0f                                                                       │
│  Final Output: Here is the market research analysis for the **sci-fi thriller genre**, using the MCP market     │
│  data and tailored to **“Echoes of Tomorrow”**.                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Sci-Fi Thriller Market Research                                                                              │
│                                                                                                                 │
│  ## 1) Current Market Performance of Sci-Fi Films                                                               │
│                                                                                                                 │
│  The sci-fi genre is performing strongly across theatrical and streaming channels.                              │
│                                                                                                                 │
│  ### Key market indicators                                                                                      │
│  - **Average box office:** **$380M**                                                                            │
│  - **Streaming performance:** **High**                                                                          │
│  - **Completion rate on streaming:** **88%**                                                                    │
│                                                                                                                 │
│  ### What this means                                                                                            │
│  - Sci-fi remains one of the most commercially resilient genres, especially when the concept is high-concept    │
│  and easy to market.                                                                                            │
│  - The strong streaming completion rate suggests audiences are not only clicking into sci-fi titles, but        │
│  sticking with them through the end. That is a good signal for:                                                 │
│    - franchise potential                                                                                        │
│    - post-theatrical demand                                                                                     │
│    - strong word-of-mouth                                                                                       │
│    - replay value                                                                                               │
│                                                                                                                 │
│  ### Market interpretation for a sci-fi thriller like “Echoes of Tomorrow”                                      │
│  Given the excerpt’s emphasis on **suspense, mystery, psychological dread, and revelation**, this project sits  │
│  in a commercially attractive lane:                                                                             │
│  - It has the **genre familiarity** of sci-fi         

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 9: View Final Results

Let's examine the consolidated output from all agents.

In [22]:
print("📋 FINAL PRODUCTION REPORT")
print("=" * 80)
print(result)
print("=" * 80)

📋 FINAL PRODUCTION REPORT
Here is the market research analysis for the **sci-fi thriller genre**, using the MCP market data and tailored to **“Echoes of Tomorrow”**.

---

# Sci-Fi Thriller Market Research

## 1) Current Market Performance of Sci-Fi Films

The sci-fi genre is performing strongly across theatrical and streaming channels.

### Key market indicators
- **Average box office:** **$380M**
- **Streaming performance:** **High**
- **Completion rate on streaming:** **88%**

### What this means
- Sci-fi remains one of the most commercially resilient genres, especially when the concept is high-concept and easy to market.
- The strong streaming completion rate suggests audiences are not only clicking into sci-fi titles, but sticking with them through the end. That is a good signal for:
  - franchise potential
  - post-theatrical demand
  - strong word-of-mouth
  - replay value

### Market interpretation for a sci-fi thriller like “Echoes of Tomorrow”
Given the excerpt’s emphasis on 

## Step 10: Implementing Observability

Observability helps you monitor and debug your agents. Let's add logging and tracking capabilities.

**Key Observability Features:**
1. Execution time tracking
2. Tool usage monitoring
3. Error logging
4. Agent decision tracking

In [23]:
import time
import json
from datetime import datetime

class CrewObserver:
    """Custom observability class for monitoring CrewAI execution"""

    def __init__(self):
        self.logs = []
        self.start_time = None
        self.end_time = None

    def log_event(self, event_type: str, agent_name: str, details: dict):
        """Log an event during crew execution"""
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'event_type': event_type,
            'agent': agent_name,
            'details': details
        }
        self.logs.append(log_entry)

    def start_monitoring(self):
        """Start monitoring crew execution"""
        self.start_time = time.time()
        self.log_event('CREW_START', 'System', {'message': 'Crew execution started'})

    def end_monitoring(self):
        """End monitoring and calculate metrics"""
        self.end_time = time.time()
        duration = self.end_time - self.start_time
        self.log_event('CREW_END', 'System', {
            'message': 'Crew execution completed',
            'duration_seconds': round(duration, 2)
        })

    def get_metrics(self):
        """Generate execution metrics"""
        if not self.end_time:
            return "Monitoring still in progress"

        duration = self.end_time - self.start_time
        agent_events = {}

        for log in self.logs:
            agent = log['agent']
            if agent != 'System':
                agent_events[agent] = agent_events.get(agent, 0) + 1

        metrics = f"""\n📊 EXECUTION METRICS\n{'='*60}
⏱️  Total Duration: {duration:.2f} seconds
📝 Total Events Logged: {len(self.logs)}
🤖 Agent Activity:"""

        for agent, count in agent_events.items():
            metrics += f"\n   • {agent}: {count} events"

        return metrics

    def display_logs(self, filter_agent=None):
        """Display execution logs"""
        print(f"\n📋 EXECUTION LOGS ({len(self.logs)} entries)")
        print("=" * 80)

        for log in self.logs:
            if filter_agent and log['agent'] != filter_agent:
                continue

            timestamp = log['timestamp'].split('T')[1].split('.')[0]
            print(f"[{timestamp}] {log['event_type']:15} | {log['agent']:25} | {log['details']}")

# Initialize observer
observer = CrewObserver()
print("✅ Observability system initialized")

✅ Observability system initialized


## Step 11: Run Crew with Observability

Now let's run our crew again with full observability enabled.

In [24]:
# Create a new task with observability hooks
observer.start_monitoring()

# Log agent initialization
observer.log_event('AGENT_INIT', 'Script Analyst', {'tools': ['sentiment_analyzer', 'soundtrack_recommender']})
observer.log_event('AGENT_INIT', 'Budget Manager', {'tools': ['budget_calculator']})
observer.log_event('AGENT_INIT', 'Market Researcher', {'tools': ['market_data_mcp']})

# New movie concept for second run
new_concept = """
Title: "The Last Symphony"
Genre: Drama
Logline: An elderly composer, losing her memory to dementia, races to complete her
final masterpiece while her estranged daughter learns to understand her through music.

Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across
the piano keys. The wonderful melody emerges from the fog of her fading memory, a
triumph over the terrible disease that threatens to steal her life's work."
"""

# Create new task
observed_task = Task(
    description=f"""Analyze this movie concept:
    {new_concept}

    Perform sentiment analysis and recommend soundtrack genres.""",
    expected_output="Sentiment analysis and soundtrack recommendations",
    agent=script_analyst,
)

# Log task start
observer.log_event('TASK_START', 'Script Analyst', {'task': 'Script analysis'})

# Create a smaller crew for demonstration
observed_crew = Crew(
    agents=[script_analyst],
    tasks=[observed_task],
    process=Process.sequential,
    verbose=False,  # Less verbose for cleaner output
)

print("🔍 Running crew with observability enabled...\n")

# Execute
result_observed = await observed_crew.kickoff_async()

# Log completion
observer.log_event('TASK_COMPLETE', 'Script Analyst', {'status': 'success'})
observer.end_monitoring()

print("\n✅ Execution complete with observability tracking!")

🔍 Running crew with observability enabled...



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Task: Analyze this movie concept:                                                                              │
│                                                                                                                 │
│  Title: "The Last Symphony"                                                                                     │
│  Genre: Drama                                                                                                   │
│  Logline: An elderly composer, losing her memory to dementia, races to complete her                             │
│  final masterpiece while her estranged daughter learns to understand her through music.                         │
│                                                                                                                 │
│  Script Excerpt: "Each note brings both joy and sadness as Margaret's fingers dance across                      │
│  the piano keys. The wonderful melody emerges from the fog of her fading memory, a                              │
│  triumph over the terrible disease that threatens to steal her life's work."                                    │
│                                                                                                                 │
│                                                                                                                 │
│      Perform sentiment analysis and recommend soundtrack genres.                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Script Analyst                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Sentiment Analysis**                                                                                         │
│                                                                                                                 │
│  - **Overall tone:** Uplifting, bittersweet, and deeply emotional                                               │
│  - **Positive tone:** 60%                                                                                       │
│  - **Negative tone:** 40%                                                                                       │
│  - **Suspenseful tone:** 0%                                                                                     │
│  - **Dominant emotions:** Hope, sadness, tenderness, triumph                                                    │
│                                                                                                                 │
│  **Interpretation:**                                                                                            │
│  The excerpt carries a poignant emotional contrast: joy in Margaret’s creative expression and sorrow in the     │
│  reality of dementia. The language frames her music as both a personal victory and a fragile struggle against   │
│  memory loss. This gives the story a moving, human-centered emotional core.                                     │
│                                                                                                                 │
│  **Soundtrack Recommendations**                                                                                 │
│                                                                                                                 │
│  For a **drama** with an **emotional** mood, the strongest soundtrack genres are:                               │
│                                                                                                                 │
│  1. **Piano Solo**                                                                                              │
│  2. **String Quartet**                                                                                          │
│  3. **Acoustic Guitar**                                                                                         │
│                                                                                                                 │
│  **Best fit for this concept:**                                                                                 │
│  - **Piano Solo** for Margaret’s inner world, memory, and final composition                                     │
│  - **String Quartet** for emotional depth, family tension, and dramatic release                                 │
│  - **Acoustic Guitar** for intimate, reflective scenes between mother and daughter                              │
│                                                                                                                 │
│  **Recommended musical approach:**                                                                              │
│  A **piano-led score with subtle string accompaniment** would best capture the film’s bittersweet beauty. Use   │
│  delicate, recurring motifs to mirror fading memory, and let the music gradually swell into richer              │
│  orchestration as the daughter understands her mother t


✅ Execution complete with observability tracking!


## 📈 Step 12: Analyze Observability Data

Let's examine the metrics and logs collected during execution.

In [25]:
# Display metrics
print(observer.get_metrics())

# Display all logs
observer.display_logs()

# Show result
print("\n📋 AGENT OUTPUT")
print("=" * 80)
print(result_observed)
print("=" * 80)


📊 EXECUTION METRICS
⏱️  Total Duration: 3.43 seconds
📝 Total Events Logged: 7
🤖 Agent Activity:
   • Script Analyst: 3 events
   • Budget Manager: 1 events
   • Market Researcher: 1 events

📋 EXECUTION LOGS (7 entries)
[07:52:04] CREW_START      | System                    | {'message': 'Crew execution started'}
[07:52:04] AGENT_INIT      | Script Analyst            | {'tools': ['sentiment_analyzer', 'soundtrack_recommender']}
[07:52:04] AGENT_INIT      | Budget Manager            | {'tools': ['budget_calculator']}
[07:52:04] AGENT_INIT      | Market Researcher         | {'tools': ['market_data_mcp']}
[07:52:04] TASK_START      | Script Analyst            | {'task': 'Script analysis'}
[07:52:07] TASK_COMPLETE   | Script Analyst            | {'status': 'success'}
[07:52:07] CREW_END        | System                    | {'message': 'Crew execution completed', 'duration_seconds': 3.43}

📋 AGENT OUTPUT
**Sentiment Analysis**

- **Overall tone:** Uplifting, bittersweet, and deeply emotiona

## Step 13: Advanced Observability - Export Logs

In production, you'd want to export logs for analysis. Here's how to save them.

In [26]:
import json

# Export logs to JSON
def export_logs(observer, filename='crew_execution_logs.json'):
    """Export observability logs to a JSON file"""
    log_data = {
        'execution_summary': {
            'start_time': datetime.fromtimestamp(observer.start_time).isoformat(),
            'end_time': datetime.fromtimestamp(observer.end_time).isoformat(),
            'duration_seconds': round(observer.end_time - observer.start_time, 2),
            'total_events': len(observer.logs)
        },
        'events': observer.logs
    }

    with open(filename, 'w') as f:
        json.dump(log_data, f, indent=2)

    return f"✅ Logs exported to {filename}"

# Export the logs
print(export_logs(observer))

# Display a sample of the JSON structure
print("\n📄 Sample JSON structure:")
sample = {
    'execution_summary': 'metadata about the run',
    'events': [
        {
            'timestamp': '2025-11-30T10:30:45.123456',
            'event_type': 'TASK_START',
            'agent': 'Script Analyst',
            'details': {'task': 'Script analysis'}
        }
    ]
}
print(json.dumps(sample, indent=2))

✅ Logs exported to crew_execution_logs.json

📄 Sample JSON structure:
{
  "execution_summary": "metadata about the run",
  "events": [
    {
      "timestamp": "2025-11-30T10:30:45.123456",
      "event_type": "TASK_START",
      "agent": "Script Analyst",
      "details": {
        "task": "Script analysis"
      }
    }
  ]
}


## 🎯 CHALLENGE: Build Your Own Movie Production Crew!

Now it's your turn to apply what you've learned! Complete the following challenge:

### 🎬 Challenge Description

Create a **Movie Casting Assistant** crew that helps with casting decisions for a new film.

### Requirements:

1. **Create TWO custom tools:**
   - `actor_availability_checker`: Takes an actor name and date range, returns if they're available (simulate with logic)
   - `role_matcher`: Takes character description and returns suggested actor traits (age range, experience level, acting style)

2. **Create ONE MCP-style tool:**
   - `fetch_actor_database`: Simulates querying a database of actors with their filmography, awards, and availability

3. **Create TWO agents:**
   - **Casting Director**: Uses role_matcher and fetch_actor_database
   - **Schedule Coordinator**: Uses actor_availability_checker

4. **Define tasks:**
   - Task 1: Find suitable actors for a lead role (give a character description)
   - Task 2: Check availability of suggested actors for specific shooting dates

5. **Add observability:**
   - Use the CrewObserver to track execution
   - Export logs at the end

### 💡 Bonus Points:
- Make your character descriptions creative and unique
- Add personality to your agents' backstories
- Include error handling in your tools
- Create a visualization of the observability data

### Sample Character (feel free to create your own!):
```
Character: Captain Zara Quinn
Description: A 35-year-old space captain with a mysterious past, known for her
quick wit and exceptional piloting skills. Must convey both strength and vulnerability.
```

### 📝 Your Code Goes Below:

Good luck! 🚀

In [27]:
# Step 1 + 2: tools
from datetime import date

# Fake actor database (made-up people)
ACTORS = {
    "Liva Ozola": {"age": 58, "experience": "veteran", "style": "quiet, naturalistic",
                   "films": ["The Amber Coast", "Winter Ferry"], "awards": ["Best Actress, Baltic Film Days 2019"],
                   "busy": [("2027-03-01", "2027-05-15")]},
    "Marta Kalnina": {"age": 63, "experience": "veteran", "style": "intense, theatre background",
                      "films": ["Salt and Iron", "Last Train to Liepaja"], "awards": [],
                      "busy": [("2027-07-01", "2027-08-15")]},
    "Ieva Sprogis": {"age": 41, "experience": "mid-career", "style": "physical, comedic timing",
                     "films": ["Sunday Market"], "awards": ["Audience Award 2024"],
                     "busy": []},
    "Anna Berg": {"age": 55, "experience": "mid-career", "style": "naturalistic, dry humour",
                  "films": ["The Keeper's Daughter", "North Wind"], "awards": [],
                  "busy": [("2027-04-20", "2027-07-30")]},
}


@tool("Actor Availability Checker")
def actor_availability_checker(actor_name: str, start_date: str, end_date: str) -> str:
    """
    Checks if an actor is free for the whole shooting period.

    Args:
        actor_name: Full name of the actor
        start_date: First shooting day, YYYY-MM-DD
        end_date: Last shooting day, YYYY-MM-DD
    """
    actor = ACTORS.get(actor_name.strip())
    if actor is None:
        return f"'{actor_name}' is not in the actor database."
    try:
        start, end = date.fromisoformat(start_date), date.fromisoformat(end_date)
    except ValueError:
        return "Dates must be in YYYY-MM-DD format."
    if end < start:
        return "End date is before start date."

    for busy_from, busy_to in actor["busy"]:
        b_from, b_to = date.fromisoformat(busy_from), date.fromisoformat(busy_to)
        if start <= b_to and b_from <= end:  # the two periods overlap
            return f"{actor_name} is NOT available: booked {busy_from} to {busy_to}."
    return f"{actor_name} is available from {start_date} to {end_date}."


@tool("Role Matcher")
def role_matcher(character_description: str) -> str:
    """
    Turns a character description into the actor traits to look for
    (age range, experience level, acting style).

    Args:
        character_description: Free text about the character
    """
    text = character_description.lower()
    ages = [int(w) for w in re.findall(r"\b(\d{2})-year-old", text)]
    age_range = f"{ages[0] - 5}-{ages[0] + 5}" if ages else "any"

    experience = "veteran" if (ages and ages[0] >= 50) or "lead" in text else "mid-career"
    style = []
    if any(w in text for w in ["quiet", "lonely", "silent", "grief"]):
        style.append("quiet, naturalistic")
    if any(w in text for w in ["funny", "wit", "humour", "humor"]):
        style.append("dry humour")
    if any(w in text for w in ["rage", "intense", "violent"]):
        style.append("intense")
    return (f"Suggested traits -> age range: {age_range}, experience: {experience}, "
            f"style: {', '.join(style) or 'open'}")


@tool("Actor Database MCP")
def fetch_actor_database(min_age: int, max_age: int) -> str:
    """
    Queries the actor database (simulated MCP service) for actors in an age range.
    Returns filmography, awards, experience and acting style.

    Args:
        min_age: Lowest age
        max_age: Highest age
    """
    rows = []
    for name, a in ACTORS.items():
        if min_age <= a["age"] <= max_age:
            rows.append(f"- {name}, {a['age']}, {a['experience']}, style: {a['style']}; "
                        f"films: {', '.join(a['films'])}; awards: {', '.join(a['awards']) or 'none'}")
    if not rows:
        return f"No actors between {min_age} and {max_age}."
    return "Actors found:\n" + "\n".join(rows)


print(role_matcher.run("A 60-year-old lighthouse keeper, quiet and lonely, with a dry wit"))
print(fetch_actor_database.run(min_age=55, max_age=65))
print(actor_availability_checker.run(actor_name="Anna Berg", start_date="2027-05-01", end_date="2027-06-15"))

Suggested traits -> age range: 55-65, experience: veteran, style: quiet, naturalistic, dry humour
Actors found:
- Liva Ozola, 58, veteran, style: quiet, naturalistic; films: The Amber Coast, Winter Ferry; awards: Best Actress, Baltic Film Days 2019
- Marta Kalnina, 63, veteran, style: intense, theatre background; films: Salt and Iron, Last Train to Liepaja; awards: none
- Anna Berg, 55, mid-career, style: naturalistic, dry humour; films: The Keeper's Daughter, North Wind; awards: none
Anna Berg is NOT available: booked 2027-04-20 to 2027-07-30.


In [28]:
# Step 3: agents
casting_director = Agent(
    role="Casting Director",
    goal="Rank the actors in the database for a lead role, best match first",
    backstory="""You cast small European drama films. You care more about the right face and
    voice than about fame, and you always explain your choice in one or two sentences.""",
    tools=[role_matcher, fetch_actor_database],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

schedule_coordinator = Agent(
    role="Schedule Coordinator",
    goal="Check which suggested actors are free for the whole shooting period",
    backstory="""You are the production's calendar person. You check every date and
    never assume someone is free without looking it up.""",
    tools=[actor_availability_checker],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

# Step 4: tasks
character = """
Character: Velta, the last keeper of a lighthouse on the Kurzeme coast
Description: A 60-year-old woman who has lived alone at the lighthouse for twenty years.
Quiet and lonely, but with a dry wit that comes out when a young surveyor arrives to
close the lighthouse down. Lead role.
"""
shooting = ("2027-05-01", "2027-06-15")

# The full ranked list goes on: with only a top 2, both picks were busy and nobody got cast
task_find = Task(
    description=f"""Find suitable actors for this lead role:
    {character}
    First use the Role Matcher, then query the actor database with the age range it gives.
    Rank ALL actors that fit, best match first, with one reason each.""",
    expected_output="A ranked list of actor names with a short reason for each",
    agent=casting_director,
)

task_schedule = Task(
    description=f"""Go through the Casting Director's ranked list in order and check each actor's
    availability for the whole shooting period {shooting[0]} to {shooting[1]}.
    Recommend the highest ranked actor who is free, and say who was skipped and why.""",
    expected_output="Availability per actor and one final casting choice",
    agent=schedule_coordinator,
    context=[task_find],
)
print("✅ 2 agents and 2 tasks ready")

✅ 2 agents and 2 tasks ready


In [29]:
# Step 5: observability
casting_observer = CrewObserver()
casting_observer.start_monitoring()
casting_observer.log_event("AGENT_INIT", "Casting Director", {"tools": ["role_matcher", "fetch_actor_database"]})
casting_observer.log_event("AGENT_INIT", "Schedule Coordinator", {"tools": ["actor_availability_checker"]})

# Step 6: crew
casting_crew = Crew(
    agents=[casting_director, schedule_coordinator],
    tasks=[task_find, task_schedule],
    process=Process.sequential,
    verbose=True,
    # called after every task, so the log gets the real finish time of each one
    task_callback=lambda out: casting_observer.log_event(
        "TASK_COMPLETE", out.agent, {"summary": out.raw[:80]}),
)

casting_observer.log_event("TASK_START", "Casting Director", {"task": "find actors"})
casting_result = await casting_crew.kickoff_async()

casting_observer.end_monitoring()

c:\Users\artem.lubkivskyi\Documents\GitHub\Accenture\Day 14\.venv-crewai\Lib\site-packages\pydantic\main.py:250: UserWarning: A function callback cannot be saved in a checkpoint; it is dropped and will not run after a restore. Use a module-level named function instead.
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 435ec2c3-55c6-46db-9e86-7e23f79209ec                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Find suitable actors for this lead role:                                                                 │
│                                                                                                                 │
│  Character: Velta, the last keeper of a lighthouse on the Kurzeme coast                                         │
│  Description: A 60-year-old woman who has lived alone at the lighthouse for twenty years.                       │
│  Quiet and lonely, but with a dry wit that comes out when a young surveyor arrives to                           │
│  close the lighthouse down. Lead role.                                                                          │
│                                                                                                                 │
│      First use the Role Matcher, then query the actor database with the age range it gives.                     │
│      Rank ALL actors that fit, best match first, with one reason each.                                          │
│  ID: 49da9046-1ba2-49a0-ba42-1350763332d2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Casting Director                                                                                        │
│                                                                                                                 │
│  Task: Find suitable actors for this lead role:                                                                 │
│                                                                                                                 │
│  Character: Velta, the last keeper of a lighthouse on the Kurzeme coast                                         │
│  Description: A 60-year-old woman who has lived alone at the lighthouse for twenty years.                       │
│  Quiet and lonely, but with a dry wit that comes out when a young surveyor arrives to                           │
│  close the lighthouse down. Lead role.                                                                          │
│                                                                                                                 │
│      First use the Role Matcher, then query the actor database with the age range it gives.                     │
│      Rank ALL actors that fit, best match first, with one reason each.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: role_matcher                                                                                             │
│  Args: {'character_description': 'Velta, the last keeper of a lighthouse on the Kurzeme coast. A 60-year-old    │
│  woman who has lived alone at the lighthouse for twenty years. Quiet and lonely, but with a dry wit...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: role_matcher                                                                                             │
│  Output: Suggested traits -> age range: 55-65, experience: veteran, style: quiet, naturalistic, dry humour      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (14 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Schedule Coordinator                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I checked the ranked list in order for the full shooting period **2027-05-01 to 2027-06-15**:                  │
│                                                                                                                 │
│  1. **Liva Ozola** — **Not available**                                                                          │
│     - Booked **2027-03-01 to 2027-05-15**                                                                       │
│     - She overlaps the shooting period, so she cannot do the full run.                                          │
│                                                                                                                 │
│  2. **Anna Berg** — **Not available**                                                                           │
│     - Booked **2027-04-20 to 2027-07-30**                                                                       │
│     - She overlaps the entire shooting period, so she cannot do the full run.                                   │
│                                                                                                                 │
│  3. **Marta Kalnina** — **Available**                                                                           │
│     - Free for **2027-05-01 to 2027-06-15**                                                                     │
│                                                                                                                 │
│  **Final casting choice: Marta Kalnina**                                                                        │
│                                                                                                                 │
│  **Skipped and why:**                                                                                           │
│  - **Liva Ozola** — unavailable due to a booking ending on **2027-05-15**                                       │
│  - **Anna Berg** — unavailable due to a booking spanning **2027-04-20 to 2027-07-30**                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Go through the Casting Director's ranked list in order and check each actor's                            │
│      availability for the whole shooting period 2027-05-01 to 2027-06-15.                                       │
│      Recommend the highest ranked actor who is free, and say who was skipped and why.                           │
│  Agent: Schedule Coordinator                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 435ec2c3-55c6-46db-9e86-7e23f79209ec                                                                       │
│  Final Output: I checked the ranked list in order for the full shooting period **2027-05-01 to 2027-06-15**:    │
│                                                                                                                 │
│  1. **Liva Ozola** — **Not available**                                                                          │
│     - Booked **2027-03-01 to 2027-05-15**                                                                       │
│     - She overlaps the shooting period, so she cannot do the full run.                                          │
│                                                                                                                 │
│  2. **Anna Berg** — **Not available**                                                                           │
│     - Booked **2027-04-20 to 2027-07-30**                                                                       │
│     - She overlaps the entire shooting period, so she cannot do the full run.                                   │
│                                                                                                                 │
│  3. **Marta Kalnina** — **Available**                                                                           │
│     - Free for **2027-05-01 to 2027-06-15**                                                                     │
│                                                                                                                 │
│  **Final casting choice: Marta Kalnina**                                                                        │
│                                                                                                                 │
│  **Skipped and why:**                                                                                           │
│  - **Liva Ozola** — unavailable due to a booking ending on **2027-05-15**                                       │
│  - **Anna Berg** — unavailable due to a booking spanning **2027-04-20 to 2027-07-30**                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
# Step 7: results, metrics, log export
print("🎬 CASTING RESULT")
print("=" * 80)
print(casting_result.raw)
print("=" * 80)

print(casting_observer.get_metrics())
casting_observer.display_logs()
print(export_logs(casting_observer, "casting_crew_logs.json"))


🎬 CASTING RESULT
I checked the ranked list in order for the full shooting period **2027-05-01 to 2027-06-15**:

1. **Liva Ozola** — **Not available**
   - Booked **2027-03-01 to 2027-05-15**
   - She overlaps the shooting period, so she cannot do the full run.

2. **Anna Berg** — **Not available**
   - Booked **2027-04-20 to 2027-07-30**
   - She overlaps the entire shooting period, so she cannot do the full run.

3. **Marta Kalnina** — **Available**
   - Free for **2027-05-01 to 2027-06-15**

**Final casting choice: Marta Kalnina**

**Skipped and why:**
- **Liva Ozola** — unavailable due to a booking ending on **2027-05-15**
- **Anna Berg** — unavailable due to a booking spanning **2027-04-20 to 2027-07-30**

📊 EXECUTION METRICS
⏱️  Total Duration: 8.36 seconds
📝 Total Events Logged: 7
🤖 Agent Activity:
   • Casting Director: 3 events
   • Schedule Coordinator: 2 events

📋 EXECUTION LOGS (7 entries)
[07:52:07] CREW_START      | System                    | {'message': 'Crew execution s

### What I noticed

- My first version asked the Casting Director for the top 2 only. Both were busy during the shoot, and the Schedule Coordinator could only answer "cast nobody", because in a sequential crew the second task cannot go back and ask for more names. With a ranked list of all matches it skipped Liva Ozola and Anna Berg and picked Marta Kalnina, the third one.
- `task_callback` gives each task its real finish time in the log (director about 5 s, coordinator about 3 s, 8.36 s total).
- The course `CrewObserver` only logs what you write into it by hand. For example it shows a "Budget Manager" event in a crew that had only the Script Analyst.
- The keyword sentiment tool gave "Echoes of Tomorrow" 0% negative, because "tragic" is not "tragedy" and "terror" is not in the list. The budget numbers are right ($28,980,000 for 60 days and 85 people, I checked by hand).

## 🎉 Congratulations!

You've completed the CrewAI practice notebook! You've learned:

✅ How to create custom tools for agents  
✅ How to implement MCP-style tools for external integrations  
✅ How to configure and use an Azure OpenAI model  
✅ How to create specialized agents with specific roles  
✅ How to define and orchestrate tasks  
✅ How to implement observability and monitoring  
✅ How to export and analyze execution logs  

### 🔗 Useful Resources:

- [CrewAI Documentation](https://docs.crewai.com/)
- [CrewAI LLM Configuration](https://docs.crewai.com/en/concepts/llms)
- [CrewAI Tools](https://docs.crewai.com/en/concepts/tools)
- [Azure OpenAI v1 API](https://learn.microsoft.com/azure/ai-foundry/openai/api-version-lifecycle)

Happy coding! 🚀🎬